# v9_b Step 5c: Assign candidate-filter and authentic-standard annotations to v9_b reaction features

This notebook adds two columns to the existing v9_b Step 5b DnHz and DnCl annotation tables. It uses three Step 2 annotation tables under `results/step2_candidate_filter_annotation/`.

**Rules**

- Preserve every Step 5b row and original column; annotate both `M0,M+H+` string forms (`M0,M+H+` and `['M0,M+H+']`). Other ions receive `NA` in the new columns because they are outside this assignment.
- Expand each pellets/supernatant source feature ID and its paired `(enzyme, substrate)` evidence. Join by `(steroid, derivatization, matrix, enzyme, original feature ID)`, never by the merged `id` alone.
- `parent_candidate_annotation` is `True` when *any* corresponding 03 record is True; otherwise `False` for target rows.
- `authentic_standard_identification` contains the deduplicated union of final 02 `authentic_standard_name` values across all corresponding sides, steroids and enzymes. If no standard is identified, write `NA`; when multiple standards remain ambiguous, retain every name separated by `; `.
- 01 is used to verify the parent MS2 flag copied into 03. The original Step 5b tables and the source candidate-filter results are unchanged.

In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
import ast
import csv


def locate_v9_b():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents, cwd / 'Shuzhao_result' / 'v9_b'):
        if (candidate / 'results' / 'step2_candidate_filter_annotation' / '03_CYP_steroid_feature_annotations.tsv').is_file():
            return candidate
    raise FileNotFoundError('Could not locate Shuzhao_result/v9_b/results/step2_candidate_filter_annotation')


BASE = locate_v9_b()
ANNOTATION_INPUT = BASE / 'results' / 'step2_candidate_filter_annotation'
NETWORK_INPUT = BASE / 'results' / 'step5b_combined_net_anno'
OUTPUT = BASE / 'results' / 'step5c_candidate_annotations'
CANDIDATE_KEY_FIELDS = ('steroid', 'derivatization', 'matrix', 'enzyme', 'unlabeled_feature')
NEW_COLUMNS = ('authentic_standard_identification', 'parent_candidate_annotation')


def read_tsv(path):
    with path.open(newline='', encoding='utf-8') as handle:
        reader = csv.DictReader(handle, delimiter='\t')
        return reader.fieldnames, list(reader)


def candidate_key(row):
    return (row['steroid'].strip().lower(), row['derivatization'].strip(),
            row['matrix'].strip(), row['enzyme'].strip(), row['unlabeled_feature'].strip())


h01, rows01 = read_tsv(ANNOTATION_INPUT / '01_parent_candidate_comparison.tsv')
h02, rows02 = read_tsv(ANNOTATION_INPUT / '02_authentic_standard_annotation.tsv')
h03, rows03 = read_tsv(ANNOTATION_INPUT / '03_CYP_steroid_feature_annotations.tsv')
assert all(set(CANDIDATE_KEY_FIELDS) <= set(h) for h in (h01, h02, h03))
by01 = {candidate_key(row): row for row in rows01}
by03 = {candidate_key(row): row for row in rows03}
assert len(by01) == len(rows01) and len(by03) == len(rows03), 'Duplicate candidate key'
assert set(by01) == set(by03), '01 and 03 candidate key sets differ'
assert all(by01[key]['ms2_supported_any_nce'] == by03[key]['parent_candidate_annotation']
           for key in by03), '01 and 03 parent flags differ'

standard_names = defaultdict(set)
for row in rows02:
    key = candidate_key(row)
    assert key in by03, f'Standard key missing from 03: {key}'
    name = row['authentic_standard_name'].strip()
    assert name, f'Empty standard name for {key}'
    standard_names[key].add(name)
assert all((by03[key]['authentic_standard_identification'] == 'True') == bool(standard_names[key])
           for key in by03), '03 standard flag and 02 names differ'
print('Copied candidate tables:', len(rows01), '01 rows;', len(rows02), '02 rows;', len(rows03), '03 rows')
print('01/02/03 key and flag consistency checks passed.')

## Expand the original side IDs and assign row-level evidence

The supernatant ID field can contain more than one semicolon-separated ID. Its `bioreactions_supernatant` value is a parallel list: the evidence at position *i* belongs to the ID at position *i*. This preserves the original steroid and enzyme context of each source feature.

In [ ]:
def split_ids(text):
    value = str(text or '').strip()
    return [] if value in ('', 'NA', 'None', 'nan') else [item.strip() for item in value.split(';') if item.strip()]


def parse_literal(text):
    value = str(text or '').strip()
    return None if value in ('', 'NA', 'None', 'nan') else ast.literal_eval(value)


def is_primary_m0h(text):
    value = str(text or '').strip()
    if value == 'M0,M+H+':
        return True
    if value.startswith('['):
        parsed = ast.literal_eval(value)
        return parsed == ['M0,M+H+']
    return False


def pair_keys(row, derivatization):
    keys = set()
    pellet_ids = split_ids(row['feature_id_pellets'])
    pellet_pairs = parse_literal(row['bioreactions_pellets'])
    assert len(pellet_ids) <= 1
    if pellet_ids:
        assert isinstance(pellet_pairs, list)
        for enzyme, substrate in pellet_pairs:
            keys.add((str(substrate).lower(), derivatization, 'pellets',
                      str(enzyme), pellet_ids[0]))
    else:
        assert pellet_pairs is None

    super_ids = split_ids(row['feature_id_supernatant'])
    super_groups = parse_literal(row['bioreactions_supernatant'])
    if super_ids:
        assert isinstance(super_groups, list) and len(super_groups) == len(super_ids), row['id']
        for feature_id, group in zip(super_ids, super_groups):
            assert isinstance(group, list)
            for enzyme, substrate in group:
                keys.add((str(substrate).lower(), derivatization, 'supernatant',
                          str(enzyme), feature_id))
    else:
        assert super_groups is None
    assert keys, f'No source candidate keys for {derivatization}:{row["id"]}'
    return keys


def write_tsv(path, fieldnames, rows):
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, delimiter='\t')
        writer.writeheader()
        writer.writerows(rows)


OUTPUT.mkdir(parents=True, exist_ok=True)
run_summary = []
for derivatization in ('DnHz', 'DnCl'):
    source = NETWORK_INPUT / f'rxn_annotated_{derivatization}_features.tsv'
    source_columns, original_rows = read_tsv(source)
    assert not set(NEW_COLUMNS).intersection(source_columns), 'New column already exists in source table'
    seen_keys = set()
    updated_rows = []
    target_rows = 0
    parent_true = 0
    standard_named = 0
    ambiguous_standards = []
    source_side_ids = set()
    for row in original_rows:
        keys = pair_keys(row, derivatization)
        assert all(key in by03 for key in keys), f'03 join miss: {keys - set(by03)}'
        seen_keys.update(keys)
        for steroid, deriv, matrix, enzyme, feature_id in keys:
            source_side_ids.add((matrix, feature_id))
        updated = row.copy()
        if is_primary_m0h(row['ion_relation']):
            target_rows += 1
            names = sorted({name for key in keys for name in standard_names[key]}, key=str.casefold)
            parent_flag = any(by03[key]['parent_candidate_annotation'] == 'True' for key in keys)
            updated['authentic_standard_identification'] = '; '.join(names) if names else 'NA'
            updated['parent_candidate_annotation'] = 'True' if parent_flag else 'False'
            parent_true += parent_flag
            standard_named += bool(names)
            if len(names) > 1:
                ambiguous_standards.append((row['id'], names))
        else:
            updated['authentic_standard_identification'] = 'NA'
            updated['parent_candidate_annotation'] = 'NA'
        updated_rows.append(updated)

    expected_keys = {key for key in by03 if key[1] == derivatization}
    assert seen_keys == expected_keys, f'Full key coverage mismatch: {derivatization}'
    assert all(all(a[col] == b[col] for col in source_columns)
               for a, b in zip(original_rows, updated_rows)), 'Original columns changed'
    assert len(updated_rows) == len(original_rows)
    output = OUTPUT / f'rxn_annotated_{derivatization}_features_with_candidate_annotations.tsv'
    write_tsv(output, [*source_columns, *NEW_COLUMNS], updated_rows)
    # Re-read the deliverable to verify row order, original values, and the two new columns.
    written_columns, written_rows = read_tsv(output)
    assert written_columns == [*source_columns, *NEW_COLUMNS]
    assert len(written_rows) == len(original_rows)
    assert all(all(before[column] == after[column] for column in source_columns)
               for before, after in zip(original_rows, written_rows))
    assert all(row['parent_candidate_annotation'] in ('True', 'False')
               for row in written_rows if is_primary_m0h(row['ion_relation']))
    summary = {'derivatization': derivatization, 'original_rows': len(original_rows),
               'm0h_rows_annotated': target_rows, 'source_side_ids': len(source_side_ids),
               'candidate_keys_matched': len(seen_keys), 'parent_true_rows': parent_true,
               'standard_named_rows': standard_named, 'ambiguous_standard_rows': len(ambiguous_standards),
               'output': str(output)}
    run_summary.append(summary)
    print(summary)
    if ambiguous_standards:
        print('Ambiguous standards retained:', ambiguous_standards)

assert sum(item['candidate_keys_matched'] for item in run_summary) == len(by03)
print('Both DnHz and DnCl annotation tables written and verified.')